# Proyecto 1 — Dashboard de Ventas y Logística
## Proceso EDA y Limpieza de Datos
**Autor:** Michael Hans Evans  
**Dataset:** Sample Superstore  
**Herramienta:** Python / pandas  

---
> Este notebook ejecuta los pasos 2 (EDA) y 3 (Limpieza) del proyecto.  


## 0. Importación de librerías

In [64]:
import pandas as pd
import numpy as np
import io
from IPython.display import display, Markdown

## 1. Carga del dataset

In [65]:
# Filas con comas en Product Name tienen quoting externo que pandas no parsea bien
with open('../data/superstore_dataset_original.csv', 'r', encoding='latin-1') as f:
    lines = f.readlines()

fixed_lines = [lines[0]]
for line in lines[1:]:
    stripped = line.strip()
    if stripped.startswith('"') and stripped.endswith('"'):
        fixed_lines.append(stripped[1:-1].replace('""', '"') + '\n')
    else:
        fixed_lines.append(line)

df_raw = pd.read_csv(io.StringIO(''.join(fixed_lines)))

print(f"Filas: {df_raw.shape[0]:,}  |  Columnas: {df_raw.shape[1]}")

Filas: 9,994  |  Columnas: 21


---
## 2 Análisis Exploratorio (EDA)



### 2.1 Completitud y confiabilidad del dataset

In [66]:
dates = pd.to_datetime(df_raw['Order Date'], format='%m/%d/%Y', errors='coerce')
periodo = f"{int(dates.dt.year.min())} — {int(dates.dt.year.max())}"
mercado = df_raw['Country'].dropna().unique()

print(f"Filas:    {df_raw.shape[0]:,}")
print(f"Columnas: {df_raw.shape[1]}")
print(f"Período:  {periodo}")
print(f"Mercado:  {', '.join(mercado)}")

Filas:    9,994
Columnas: 21
Período:  2014 — 2017
Mercado:  United States


**Observaciones:**
- Dataset histórico cerrado (sin actualizaciones futuras)
- Mercado único — limita análisis geográfico comparativo
- Fuente estructurada con esquema consistente

### 2.2 Perfilado de columnas

In [67]:
perfil = pd.DataFrame({
    'Tipo':    df_raw.dtypes.astype(str),
    'Únicos':  df_raw.nunique(),
    'Muestra': df_raw.iloc[0].astype(str).str[:30]
})

perfil

,Tipo,Únicos,Muestra
Row ID,int64,9994,1
Order ID,object,5009,CA-2016-152156
Order Date,object,1237,11/8/2016
Ship Date,object,1334,11/11/2016
Ship Mode,object,4,Second Class
Customer ID,object,793,CG-12520
Customer Name,object,793,Claire Gute
Segment,object,3,Consumer
Country,object,1,United States
City,object,531,Henderson


**Problemas detectados:**
- `Order Date` y `Ship Date`: string → debería ser datetime
- `Postal Code`: int64 → debería ser string (riesgo de perder ceros a la izquierda)
- `Country`: valor constante → columna redundante
- `Row ID`: redundante con el índice

### 2.3 Detección de valores nulos

In [68]:
nulos = df_raw.isnull().sum()
pct_nulos = (nulos / len(df_raw) * 100).round(2)

df_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje %': pct_nulos})
df_nulos = df_nulos[df_nulos['Nulos'] > 0]

print(f"Total de nulos: {nulos.sum()}")
df_nulos if not df_nulos.empty else "Sin valores nulos"

Total de nulos: 0


'Sin valores nulos'

**Conclucion:** Ninguna acción requerida — sin nulos que tratar.

### 2.4 Detección de duplicados

In [69]:
dup_exactos = df_raw.duplicated().sum()
dup_order   = df_raw.duplicated(subset=['Order ID']).sum()
dup_cliente = df_raw.duplicated(subset=['Customer ID']).sum()

print(f"Duplicados exactos:              {dup_exactos}")
print(f"Duplicados parciales (Order ID): {dup_order}")
print(f"Duplicados parciales (Customer): {dup_cliente}")

Duplicados exactos:              0
Duplicados parciales (Order ID): 4985
Duplicados parciales (Customer): 9201


**Conclucion:** Sin duplicados exactos. Los parciales son comportamiento esperado: múltiples productos por orden y múltiples órdenes por cliente.

### 2.5 Estadísticas descriptivas completas — Variables numéricas

In [70]:
num_cols = ['Sales', 'Quantity', 'Discount', 'Profit']

df_stats = df_raw[num_cols].agg(['min', 'max', 'mean', 'median', 'std', 'skew']).T
df_stats.columns = ['Mínimo', 'Máximo', 'Promedio', 'Mediana', 'Desvío', 'Skewness']
df_stats['CV %'] = (df_stats['Desvío'] / df_stats['Promedio'] * 100).round(2)
df_stats['Curtosis'] = df_raw[num_cols].kurtosis()
df_stats[['Mínimo','Máximo','Promedio','Mediana','Desvío']] = df_stats[['Mínimo','Máximo','Promedio','Mediana','Desvío']].round(2)
df_stats[['Skewness','Curtosis']] = df_stats[['Skewness','Curtosis']].round(4)

df_stats

,Mínimo,Máximo,Promedio,Mediana,Desvío,Skewness,CV %,Curtosis
Sales,0.44,22638.48,229.86,54.49,623.25,12.9728,271.14,305.3118
Quantity,1.00,14.00,3.79,3.00,2.23,1.2785,58.72,1.9919
Discount,0.00,0.80,0.16,0.20,0.21,1.6843,132.17,2.4095
Profit,-6599.98,8399.98,28.66,8.67,234.26,7.5614,817.47,397.1885


**Interpretaciones clave:**
- Profit CV = 817.47% y Sales CV = 271.14% → distribuciones extremadamente dispersas, la mediana es más representativa
- Profit Skewness = 7.56 → cola derecha larga (órdenes de profit extremadamente alto); las pérdidas existen pero el sesgo lo dominan los outliers positivos
- Sales Skewness = 12.97 → cola derecha (órdenes de alto valor)
- Discount → confirmar distribución bimodal en la siguiente sección

### 2.6 Análisis de distribución — Discount

In [71]:
discount_freq = df_raw['Discount'].value_counts().sort_index()
pd.DataFrame({
    'Frecuencia': discount_freq,
    'Porcentaje %': (discount_freq / len(df_raw) * 100).round(2)
})

,Frecuencia,Porcentaje %
Discount,,
0.00,4798,48.01
0.10,94,0.94
0.15,52,0.52
0.20,3657,36.59
0.30,227,2.27
0.32,27,0.27
0.40,206,2.06
0.45,11,0.11
0.50,66,0.66


**Interpretación:** La concentración en 0.0 y 0.2 confirma distribución bimodal. Los valores son fijos, no continuos → política de descuentos predefinida, no discrecional.

### 2.7 Identificación y validación de outliers

In [72]:
outlier_cols = ['Sales', 'Profit', 'Discount']

Q1 = df_raw[outlier_cols].quantile(0.25)
Q3 = df_raw[outlier_cols].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

outlier_inf = (df_raw[outlier_cols] < lower).sum()
outlier_sup = (df_raw[outlier_cols] > upper).sum()

pd.DataFrame({
    'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
    'Umbral inf': lower,  'Out. inf': outlier_inf,
    'Umbral sup': upper,  'Out. sup': outlier_sup,
    '% Outliers': ((outlier_inf + outlier_sup) / len(df_raw) * 100).round(2)
}).round(2)

,Q1,Q3,IQR,Umbral inf,Out. inf,Umbral sup,Out. sup,% Outliers
Sales,17.28,209.94,192.66,-271.71,0,498.93,1167,11.68
Profit,1.73,29.36,27.64,-39.72,604,70.82,1277,18.82
Discount,0.00,0.20,0.20,-0.30,0,0.50,856,8.57


**Validación de negocio:**
- Sales outliers superiores → órdenes grandes de clientes corporativos → **mantener**
- Profit outliers negativos → ventas con pérdida real → **mantener y analizar**
- Discount outliers → descuentos agresivos válidos → **mantener y analizar**

Ningún outlier fue identificado como error de carga.

### 2.8 Distribución de variables categóricas

In [73]:
cat_cols = ['Ship Mode', 'Segment', 'Region', 'Category', 'Sub-Category', 'Country']

for col in cat_cols:
    freq = df_raw[col].value_counts()
    df_cat = pd.DataFrame({
        'Frecuencia': freq,
        'Porcentaje %': (freq / len(df_raw) * 100).round(2)
    })
    display(Markdown(f"**{col}** — {df_raw[col].nunique()} valores únicos"))
    display(df_cat)

**Ship Mode** — 4 valores únicos

,Frecuencia,Porcentaje %
Ship Mode,,
Standard Class,5968,59.72
Second Class,1945,19.46
First Class,1538,15.39
Same Day,543,5.43


**Segment** — 3 valores únicos

,Frecuencia,Porcentaje %
Segment,,
Consumer,5191,51.94
Corporate,3020,30.22
Home Office,1783,17.84


**Region** — 4 valores únicos

...

,Frecuencia,Porcentaje %
Category,,
Office Supplies,6026,60.30
Furniture,2121,21.22
Technology,1847,18.48


**Sub-Category** — 17 valores únicos

,Frecuencia,Porcentaje %
Sub-Category,,
Binders,1523,15.24
Paper,1370,13.71
Furnishings,957,9.58
Phones,889,8.90
Storage,846,8.47
Art,796,7.96
Accessories,775,7.75
Chairs,617,6.17
Appliances,466,4.66


**Country** — 1 valores únicos

,Frecuencia,Porcentaje %
Country,,
United States,9994,100.0


**Hallazgos clave:**
- Country → valor constante 'United States' → candidata a eliminación
- Standard Class → 59.72% de las órdenes → modalidad dominante
- Office Supplies → 60.30% de órdenes → mayor volumen, pero ¿mayor margen?

### 2.9 Matriz de correlación — Variables numéricas

In [74]:
corr_matrix = df_raw[['Sales', 'Quantity', 'Discount', 'Profit']].corr().round(4)
corr_matrix.style.background_gradient(cmap='RdBu_r', vmin=-1, vmax=1).format('{:.4f}')

,Sales,Quantity,Discount,Profit
Sales,1.0000,0.2008,-0.0282,0.4791
Quantity,0.2008,1.0000,0.0086,0.0663
Discount,-0.0282,0.0086,1.0000,-0.2195
Profit,0.4791,0.0663,-0.2195,1.0000


In [75]:
display(Markdown("**Discount vs Profit por categoría:**"))

corr_by_cat = df_raw.groupby('Category').apply(
    lambda g: g['Discount'].corr(g['Profit']), include_groups=False
).round(4).rename('Correlación')

corr_by_cat.to_frame()

**Discount vs Profit por categoría:**

,Correlación
Category,
Furniture,-0.4838
Office Supplies,-0.2088
Technology,-0.2689


**Interpretación:**
- Discount vs Profit global: correlación negativa (-0.22) — a mayor descuento, menor profit
- La correlación varía significativamente entre categorías: Furniture muestra la más fuerte (-0.48)
- El análisis global puede enmascarar comportamientos críticos por categoría

### 2.10 Verificación de consistencia de formatos

In [76]:
order_dt = pd.to_datetime(df_raw['Order Date'], format='%m/%d/%Y', errors='coerce')
ship_dt  = pd.to_datetime(df_raw['Ship Date'], format='%m/%d/%Y', errors='coerce')

pd.DataFrame({
    'Check': ['Sales < 0', 'Discount fuera [0,1]', 'Profit < 0', 'Ship Date < Order Date'],
    'Registros': [
        (df_raw['Sales'] < 0).sum(),
        ((df_raw['Discount'] < 0) | (df_raw['Discount'] > 1)).sum(),
        (df_raw['Profit'] < 0).sum(),
        (ship_dt < order_dt).sum()
    ]
}).set_index('Check')

,Registros
Check,
Sales < 0,0
"Discount fuera [0,1]",0
Profit < 0,1871
Ship Date < Order Date,0


**Hallazgos:**
- Profit negativo presente en 1,871 registros → válido de negocio, no es error
- Sin inconsistencias en rangos ni en orden temporal

### 2.11 Identificación de columnas redundantes

In [77]:
print(f"Country: {df_raw['Country'].nunique()} valor único")
print(f"Row ID:  {df_raw['Row ID'].nunique()} valores únicos (= {len(df_raw)} filas)")

Country: 1 valor único
Row ID:  9994 valores únicos (= 9994 filas)


**Decisión:**
- `Country` → valor constante 'United States' → sin varianza, no aporta al análisis
- `Row ID` → identificador secuencial sin valor analítico

Ambas candidatas a eliminación en la etapa de limpieza.

### 2.13 Resumen de hallazgos del EDA

| # | Hallazgo | Sev. | Impacto | Acción |
|---|----------|------|---------|--------|
| 1 | Order Date y Ship Date como texto | Alta | Impide análisis temporal | Conversión a datetime |
| 2 | Profit negativo en 1,871 registros | Alta | Hallazgo crítico de negocio | Mantener y analizar |
| 3 | CV Profit = 817.47% | Alta | Media no representativa | Usar mediana |
| 4 | Sospecha pérdidas Tables/Bookcases | Alta | Relevante para H1 | Analizar en detalle |
| 5 | Discount distribución bimodal | Media | Política comercial fija | Analizar umbrales |
| 6 | Postal Code como int | Media | Riesgo pérdida ceros izquierda | Conversión a string |
| 7 | Country valor constante | Baja | Sin valor analítico | Eliminación |
| 8 | Row ID redundante | Baja | Sin valor analítico | Eliminación |

---
## 3 Limpieza y Transformación
> Las transformaciones se ejecutan en el orden correcto respetando dependencias entre pasos:  
> **1. Tipos de datos → 2. Duplicados → 3. Estandarización → 4. Eliminación → 5. Columnas calculadas**


### 3.0 Copia de trabajo

In [78]:
# Trabajamos siempre sobre una copia — nunca modificamos el raw
df = df_raw.copy()

### 3.1 Corrección de tipos de datos

In [79]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y', errors='coerce')
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%m/%d/%Y', errors='coerce')
df['Postal Code'] = (
    df['Postal Code']
    .dropna()
    .astype(int)
    .astype(str)
    .str.zfill(5)
)

print(f"Order Date:  {df['Order Date'].dtype}")
print(f"Ship Date:   {df['Ship Date'].dtype}")
print(f"Postal Code: {df['Postal Code'].dtype} → muestra: {df['Postal Code'].dropna().iloc[:3].tolist()}")

Order Date:  datetime64[ns]
Ship Date:   datetime64[ns]
Postal Code: object → muestra: ['42420', '42420', '90036']


### 3.2 Tratamiento de duplicados

In [80]:
dup_exactos = df.duplicated().sum()
print(f"Duplicados exactos: {dup_exactos}")

Duplicados exactos: 0


### 3.3 Estandarización de formatos de texto

In [81]:
text_cols = ['Ship Mode', 'Segment', 'Category', 'Sub-Category', 'State', 'City']

antes = df[text_cols].nunique()
df[text_cols] = df[text_cols].apply(lambda x: x.str.strip())
despues = df[text_cols].nunique()

print(f"Columnas con cambios tras strip: {(antes != despues).sum()} de {len(text_cols)}")

Columnas con cambios tras strip: 0 de 6


Sin inconsistencias de espacios detectadas — texto ya estandarizado en origen.

### 3.4 Tratamiento de outliers

**Decisión:** Mantener todos los outliers identificados en 2.7 — validados como comportamiento real de negocio. No se aplica eliminación, winsorización ni imputación.

### 3.5 Eliminación de columnas irrelevantes

In [82]:
antes = len(df)
df = df.drop(columns=['Country', 'Row ID'])
# Al eliminar Row ID se revela 1 duplicado exacto (misma orden, producto y valores)
df = df.drop_duplicates()

print(f"Columnas: {df.shape[1]} (eliminadas: Country, Row ID)")
print(f"Filas: {len(df):,} ({antes - len(df)} duplicado eliminado tras quitar Row ID)")

Columnas: 19 (eliminadas: Country, Row ID)
Filas: 9,993 (1 duplicado eliminado tras quitar Row ID)


### 3.6 Generación de columnas calculadas

In [83]:
df['Discount_pct'] = (df['Discount'] * 100).round(2)
df['Order_Year']    = df['Order Date'].dt.year
df['Order_Month']   = df['Order Date'].dt.month
df['Order_Quarter'] = df['Order Date'].dt.quarter
df['Profit_Margin_pct'] = (df['Profit'] / df['Sales'] * 100).round(4)
df['Shipping_Days'] = (df['Ship Date'] - df['Order Date']).dt.days

nuevas = ['Discount_pct', 'Order_Year', 'Order_Month', 'Order_Quarter', 'Profit_Margin_pct', 'Shipping_Days']
df[nuevas].describe().round(2)

,Discount_pct,Order_Year,Order_Month,Order_Quarter,Profit_Margin_pct,Shipping_Days
count,9993.00,9993.00,9993.00,9993.00,9993.00,9993.00
mean,15.62,2015.72,7.81,2.88,12.03,3.96
std,20.65,1.12,3.28,1.06,46.68,1.75
min,0.00,2014.00,1.00,1.00,-275.00,0.00
25%,0.00,2015.00,5.00,2.00,7.50,3.00
50%,20.00,2016.00,9.00,3.00,27.00,4.00
75%,20.00,2017.00,11.00,4.00,36.25,5.00
max,80.00,2017.00,12.00,4.00,50.00,7.00


### 3.7 Verificación del dataset limpio

In [84]:
pd.DataFrame({
    'Raw': [df_raw.shape[0], df_raw.shape[1], df_raw.isnull().sum().sum(), df_raw.duplicated().sum()],
    'Limpio': [df.shape[0], df.shape[1], df.isnull().sum().sum(), df.duplicated().sum()]
}, index=['Filas', 'Columnas', 'Nulos', 'Duplicados'])

,Raw,Limpio
Filas,9994,9993
Columnas,21,25
Nulos,0,0
Duplicados,0,0


In [85]:
print(f"Columnas finales ({df.shape[1]}):")
print(', '.join(df.columns))

Columnas finales (25):
Order ID, Order Date, Ship Date, Ship Mode, Customer ID, Customer Name, Segment, City, State, Postal Code, Region, Product ID, Category, Sub-Category, Product Name, Sales, Quantity, Discount, Profit, Discount_pct, Order_Year, Order_Month, Order_Quarter, Profit_Margin_pct, Shipping_Days


**Columnas eliminadas:** Country, Row ID  
**Columnas nuevas:** Discount_pct, Order_Year, Order_Month, Order_Quarter, Profit_Margin_pct, Shipping_Days

### 3.8 Exportación del dataset limpio

In [86]:
df.to_csv('superstore_clean.csv', index=False)

print(f"Exportado: superstore_clean.csv ({df.shape[0]:,} filas × {df.shape[1]} columnas)")

Exportado: superstore_clean.csv (9,993 filas × 25 columnas)


**Uso downstream:** input para análisis SQL y dashboard Plotly Dash.  


### 3.9 Documentación de decisiones

**Decisiones de limpieza:**

| # | Decisión | Justificación | Alternativa descartada |
|---|----------|---------------|------------------------|
| 1 | Fechas a datetime | Tipo incorrecto impide operaciones temporales | Mantener como string — imposibilita cálculos |
| 2 | Postal Code a string | Códigos postales no son numéricos (01040 ≠ 1040) | Mantener como int — pérdida de información |
| 3 | Eliminar Country | Varianza cero — valor constante | Mantener — innecesario |
| 4 | Eliminar Row ID | Redundante — sin valor analítico | Mantener — innecesario |
| 5 | Mantener Profit negativo | Pérdidas reales — objeto central de H1 y H2 | Eliminar/imputar — pérdida de información crítica |
| 6 | Mantener outliers Sales | Órdenes grandes de clientes corporativos | Winsorizar — distorsionaría análisis premium |
| 7 | Mantener descuentos > 50% | Política comercial — objeto central de H2 | Eliminar — son el objeto de análisis |

---
Pipeline de EDA y limpieza completado. Dataset exportado como input para análisis SQL y dashboard.